# Introducción al SIG con Python

**Primeras Jornadas de Innovación Geoespacial · UTEC · Minas, Lavalleja · 16 de octubre de 2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FelipeSBarros/taller-sig-python-utec-2026/blob/main/T1_Introduccion_SIG_Python.ipynb)

A través de ejercicios prácticos, vamos a aprender al respecto de los datos **Geoespaciales**. Vamos a aprender a cargar, explorar y visualizar algunos tipos de datos geográficos usando Python, y a conocer herramientas y librerías que permiten incorporar la programación a los flujos de trabajo habituales de los SIG.

Material producido por [Felipe Sodré Mendes Barros](https://felipesbarros.github.io/).

### ¿Cómo usar este notebook?

* Cada bloque gris es una **celda de código**: se ejecuta con el botón ▶ o con `Shift + Enter`.
* Las celdas que empiezan con **Ejercicio** están para que las completes vos. Debajo de la consigna (las líneas que empiezan con `#` son comentarios: Python no las ejecuta) escribí tu código.

### Instalando y cargando las librerías (módulos de Python)

* [geopandas](https://geopandas.org/): datos vectoriales;
* [geoplot](https://residentmario.github.io/geoplot/): visualización de datos geográficos (lo vamos a usar para los mapas de densidad);
* [rasterio](https://rasterio.readthedocs.io/): datos ráster;
* [earthpy](https://earthpy.readthedocs.io/): funciones para visualizar imágenes de satélite (composiciones RGB);
* [matplotlib](https://matplotlib.org/) y [numpy](https://numpy.org/): gráficos y cálculos con matrices.

En Colab casi todo ya viene instalado: solo hace falta instalar `geoplot` y `earthpy`.

In [ ]:
%pip install geoplot earthpy --quiet

In [ ]:
import geopandas as gpd
import geoplot as gplt
import matplotlib.pyplot as plt
import numpy as np

import rasterio
import earthpy.plot as ep

from rasterio.plot import show

### Los datos

Todos los datos del taller están en el repositorio [taller-sig-python-utec-2026](https://github.com/FelipeSBarros/taller-sig-python-utec-2026) y se leen directamente desde internet. 

In [ ]:
URL = "https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/datos/"

# Estructura de los datos geográficos

## Datos vectoriales

* Vectoriales:
  * Puntos (lon, lat);
  * Líneas [(lon¹, lat¹), (lon², lat²), (lon³, lat³)];
  * Polígonos [(lon¹, lat¹), (lon², lat²), (lon³, lat³), (lon¹, lat¹)];


Y, sobre todo, **pueden** tener una **tabla de atributos** asociada a cada observación, con información sobre ella.

* **Atributos:** mediciones o datos de una variable para cada elemento;
> Ej.: altura de un árbol, nombre de la especie, nombre de la calle, nombre del departamento, población.

<figure>
<img src="https://volaya.github.io/libro-sig/img/Primitivas_vectoriales.png">
<figcaption>Fuente: <a href="https://volaya.github.io/libro-sig/">Sistemas de Información Geográfica, un libro libre de Víctor Olaya</a></figcaption>
</figure>

## Datos matriciales

* Matriciales (ráster o superficie):
  * Imagen de una sola banda;
  * Imagen compuesta por varias bandas;

<figure>
<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fingenieriaambiental.net%2Fwp-content%2Fuploads%2Fsig-raster.jpg&f=1&nofb=1&ipt=6a97ece36bd62e73f27c6fb0a535847ea1750a21934b2eb0cb7ec564dadff93d&ipo=images">
</figure>

## Estructura de redes
* Grafos (los vemos en el taller de Redes Urbanas).

<figure>
<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fstatic.wixstatic.com%2Fmedia%2F5d1736_564e2a3e83a047849b8f4651af768f9a~mv2.png%2Fv1%2Ffill%2Fw_980%2Ch_551%2Cal_c%2Cq_90%2Cusm_0.66_1.00_0.01%2Cenc_auto%2F5d1736_564e2a3e83a047849b8f4651af768f9a~mv2.png&f=1&nofb=1&ipt=948639b6d6a6499c00ad4fff82ee0d24ecf404b879ff03f41745f4fc9eaf61be&ipo=images">
</figure>

## Formatos de archivo de datos vectoriales

### Shapefile

Es uno de los primeros formatos creados para datos vectoriales y todavía es uno de los más usados. **Está formado, como mínimo, por 4 archivos** obligatorios:

* `.shp`: guarda la geometría;
* `.dbf`: guarda la tabla de atributos;
* `.shx`: relaciona el `.shp` con el `.dbf`;
* `.prj`: contiene la información del **Sistema de Referencia de Coordenadas**.

**Cuando descargues datos geográficos en formato shapefile, asegurate de tener todos los archivos.** Por eso es muy común que se distribuyan dentro de un `.zip`, como el que vamos a usar.

**Cargando un dato vectorial en `.shp`**

Vamos a cargar los **focos de calor** detectados por satélite en Uruguay entre 2018 y 2024, para tratar de entender cómo se distribuyen en el territorio y en el tiempo.

El sistema [FIRMS](https://firms.modaps.eosdis.nasa.gov/) de la NASA publica, casi en tiempo real, los puntos donde los satélites detectan una temperatura anormalmente alta en la superficie: incendios forestales, quemas de campo o de rastrojos, pero también algunas fuentes fijas de calor, como industrias. Usamos el sensor VIIRS del satélite Suomi NPP, que tiene una resolución de 375 m. Los datos se pueden descargar libremente desde su sitio; nosotros ya los preparamos como shapefile, dentro de un `.zip`.

Para leerlo usamos la función `read_file()` de `geopandas`, a la que le pasamos la dirección del archivo. `geopandas` puede leer el `.zip` directamente, sin descomprimirlo.

In [ ]:
# Ejercicio: usá gpd.read_file() para cargar URL + "focos_firms_uy.zip" en un objeto llamado focos.
# Después escribí focos en una línea sola, para ver la tabla.


#### Geopandas

[`geopandas`](https://geopandas.org/en/stable/docs/user_guide/data_structures.html) es el módulo de Python que nos da la estructura para trabajar con datos vectoriales. Como su nombre lo indica, es una extensión de [`pandas`](https://pandas.pydata.org/) (la librería de tablas de Python) que, visualmente, agrega una columna más: la **columna de geometría**.

<img width=60% src="https://geopandas.org/en/stable/_images/dataframe.svg">

**Entendiendo el dato:**

Cada fila (*feature*, o elemento) es un foco de calor detectado por el satélite.

Como atributos (columnas) tenemos:
* `fecha`: fecha de la detección (hora de Uruguay);
* `anio`, `mes`, `hora`: año, mes (1 a 12) y hora de la detección;
* `estacion`: estación del año (`verano`, `otoño`, `invierno`, `primavera`);
* `dianoche`: si el satélite pasó de `día` o de `noche`;
* `confianza`: confianza de la detección (`baja`, `nominal` o `alta`);
* `frp`: potencia radiativa del fuego, en megavatios (cuánta energía emite: un indicador de la intensidad);
* `depto`: departamento donde cae el foco;
* `geometry`: la geometría, en este caso un punto (longitud, latitud).

### Presentando los datos como mapa

Como cargamos la capa vectorial con `geopandas`, `matplotlib` la va a tratar como un dato espacial. Así, podemos visualizarla como mapa con el método `.plot()`, sin tener que "explicarle" nada a `matplotlib`.

In [ ]:
# Ejercicio: visualizá los focos usando el método .plot()


¿Ya se reconoce la forma de Uruguay? 
Con más de 21.000 puntos, el país se dibuja solo. Pero falta algo: los límites.

### Geopackage

Es otro formato de archivo, creado por el [OGC](https://www.geopackage.org/) (Open Geospatial Consortium) para guardar y distribuir datos espaciales, principalmente vectoriales.

En **un solo archivo** `.gpkg` se pueden guardar **varias capas**, incluso con geometrías diferentes.

Además, tiene varias mejoras respecto del shapefile: guarda fechas y horas, no limita el largo del nombre de las columnas (en el shapefile, máximo 10 caracteres)...

Y no se restringe a datos vectoriales: también puede guardar datos ráster.

Vamos a cargar los límites de los departamentos de Uruguay, que están en el **geopackage** del taller.

Pero, como vimos, un geopackage puede tener más de una capa. Por eso, tenemos que indicar el nombre de la capa que queremos cargar.

Como no sabemos qué capas tiene, primero vamos a listarlas con la función `gpd.list_layers()`, indicando entre paréntesis la dirección del geopackage: `URL + "taller1_uy.gpkg"`.

In [ ]:
# Ejercicio: listá las capas del geopackage con gpd.list_layers()


**Leyendo una capa de un `.gpkg`**

`geopandas` puede leer tanto shapefiles como capas guardadas en geopackages, con la misma función `read_file()`. La diferencia es que ahora le decimos qué capa queremos, con el parámetro `layer`. Para ver qué otros formatos puede leer `geopandas`, mirá [la documentación](https://geopandas.org/en/stable/docs/user_guide/io.html).

In [ ]:
# cargando la capa "departamentos" del geopackage
deptos = gpd.read_file(URL + "taller1_uy.gpkg", layer="departamentos")
# y la capa "minas" (el área urbana de Minas), que vamos a usar más adelante
minas = gpd.read_file(URL + "taller1_uy.gpkg", layer="minas")
deptos

In [ ]:
# Ejercicio: visualizá los departamentos con .plot()


### Visualizando más de una capa

Recuerden que estamos trabajando con **capas** espaciales. Y, como en las capas de algunos programas de edición de imágenes, **el orden de las capas cambia lo que vemos**.

Para dibujar una capa sobre otra, guardamos el "lienzo" (en `matplotlib` se llama `ax`, de *axes*) del primer plot, y se lo pasamos al segundo con el parámetro `ax=`.

In [ ]:
# Ejercicio: dibujá los departamentos y, encima, los focos en color negro.
# Pista: ax = deptos.plot()  y después  focos.plot(ax=ax, color="black")


Los departamentos quedaron tapados. Podemos cambiar cómo se dibuja cada capa:
* `facecolor="none"`: sin relleno; `edgecolor`: color del borde; `linewidth`: grosor del borde;
* `color`: color de los puntos; `markersize`: tamaño de los puntos;
* `figsize=(ancho, alto)`: tamaño de la figura, en pulgadas.

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="grey", linewidth=0.5, figsize=(8, 8))
focos.plot(ax=ax, color="red", markersize=0.5)

In [ ]:
# Ejercicio: probá cambiar los colores, el grosor de los bordes y el tamaño de los puntos.
# ¿Qué pasa si dibujás primero los focos y después los departamentos (con relleno)?


### Seleccionando datos

Muchas veces no queremos todos los datos. Para quedarnos solo con algunas filas, usamos una condición entre corchetes. Por ejemplo, los focos de Lavalleja:

```python
focos_lav = focos[focos["depto"] == "LAVALLEJA"]
```

(`==` pregunta "¿es igual a?"; un solo `=` guarda un valor en una variable.)

In [ ]:
# Ejercicio: creá focos_lav con los focos de Lavalleja y dibujá un mapa de Lavalleja (sin relleno)
# con sus focos encima y la ciudad de Minas en negro.
# ¿Cuántos focos hubo en Lavalleja? Pista: len(focos_lav)


## Del dato a la visualización para la obtención del conocimiento

### Colores según una columna

También podemos pintar cada elemento según el valor de una de sus columnas, con el parámetro `column`. Si la columna es de texto (categorías), conviene agregar `legend=True` para ver la leyenda. Y con `ax.set_title()` le ponemos un título al mapa.

Probemos armar un plot usando la información que está en la tabla de atributos "estacion".

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="grey", linewidth=0.5, figsize=(8, 8))
focos.plot(ax=ax, column="estacion", markersize=0.5, legend=True)
ax.set_title("Focos de calor en Uruguay, 2018–2024, según la estación del año")
ax.set_axis_off()

In [ ]:
# Ejercicio: hacé el mismo mapa, pero coloreando por "confianza" o por "dianoche".


### Guardando el "mapa"

Como estamos usando el `pyplot` de `matplotlib`, alcanza con usar `plt.savefig()` **en la misma celda** en la que hicimos el mapa:

`plt.savefig("mapa_focos.png", dpi=150)`

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="grey", linewidth=0.5, figsize=(8, 8))
focos.plot(ax=ax, color="red", markersize=0.5)
ax.set_title("Focos de calor en Uruguay, 2018–2024")
ax.set_axis_off()
plt.savefig("mapa_focos.png", dpi=150)

---
# Del mapa al análisis: densidad de kernel

En el mapa de puntos hay tantos focos que se superponen: es difícil decir dónde hay **más**. Una de las formas más usadas de visualizar datos de puntos es con un **estimador de densidad de *kernel*** (KDE), también conocido como *heat map* o mapa de densidad. Para eso vamos a usar la función `kdeplot` (*kernel density estimate*) del paquete `geoplot`.

### Celda de rescate

Si algo de lo anterior no funcionó, ejecutá esta celda: carga todo lo que necesitamos para seguir.

In [ ]:
URL = "https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/datos/"
focos = gpd.read_file(URL + "focos_firms_uy.zip")
deptos = gpd.read_file(URL + "taller1_uy.gpkg", layer="departamentos")
lavalleja = gpd.read_file(URL + "taller1_uy.gpkg", layer="lavalleja")
minas = gpd.read_file(URL + "taller1_uy.gpkg", layer="minas")
focos_lav = focos[focos["depto"] == "LAVALLEJA"]

El módulo [`geoplot`](https://residentmario.github.io/geoplot/) es otra forma de visualizar datos geográficos, que busca facilitar todavía más el proceso.

> geoplot is a geospatial data visualization library designed for data scientists and geospatial analysts that just want to get things done.

Aunque hereda la configuración de `matplotlib`, la forma de usarlo cambia. En lugar de que el `plot` sea un método del GeoDataFrame:

```python
focos.plot()
```

pasamos a usar las [funciones](https://residentmario.github.io/geoplot/api_reference.html) de `geoplot`, pasándoles el GeoDataFrame como parámetro:

```python
import geoplot as gplt
gplt.pointplot(focos)
```

Vamos a lo que interesa: usar el [`kdeplot`](https://residentmario.github.io/geoplot/plot_references/plot_reference.html#kdeplot) para extraer más información sobre la distribución de los focos de calor en Uruguay. El parámetro `extent` define el área del mapa; usamos los límites (`total_bounds`) de los departamentos.

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="black", linewidth=0.5, figsize=(8, 8))
gplt.kdeplot(focos, ax=ax, extent=deptos.total_bounds)

In [ ]:
# total_bounds devuelve las coordenadas del rectángulo que contiene a la capa: [xmín, ymín, xmáx, ymáx]
deptos.total_bounds

### ¿Cómo interpretar el resultado?

Imaginá que cada foco "derrama" un poco de su presencia alrededor, como una campana que es más alta en el punto y baja a medida que nos alejamos. El KDE suma todas esas campanas: donde hay muchos focos cerca unos de otros, la suma es alta. Las líneas del mapa son **isolíneas de densidad**: unen lugares con la misma densidad de focos, como las curvas de nivel de un mapa topográfico. Cuanto más juntas y más "adentro" están, más concentración hay.

Ojo: el KDE muestra **dónde se concentran los eventos**, no su intensidad (para eso tendríamos la columna `frp`). Y el resultado depende del "ancho" de la campana: con campanas muy anchas todo se suaviza; con campanas muy angostas vemos cada punto.

Más información: [estimación de densidad de kernel (Wikipedia)](https://es.wikipedia.org/wiki/Estimaci%C3%B3n_de_la_densidad_de_Kernel).


<figure>
<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.paulamoraga.com%2Fbook-spatial%2Ffigures%2Fkernelplot.png&f=1&nofb=1&ipt=e63b0fc9c186eaa89c1b9a580c2c9f2e6c919a16bb00f0ceff23b55bdf8f7058&ipo=images">
</figure>

In [ ]:
# Ejercicio: hacé el KDE solo para Lavalleja: los focos de focos_lav, sobre el polígono de lavalleja
# y con extent=lavalleja.total_bounds


**Podemos cambiar algunos parámetros del gráfico:**

* `fill=True`: en lugar de isolíneas, vemos manchas de color;
* `cmap="Reds"`: la paleta de colores (otras: `"viridis"`, `"magma"`, `"YlOrRd"`...);
* `clip=deptos`: *clip* es un término técnico muy usado en SIG. En este caso, limita el resultado al polígono indicado. Se puede entender como "recortar" el análisis con otra capa de polígonos.

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="black", linewidth=0.5, figsize=(8, 8))
gplt.kdeplot(focos, ax=ax, fill=True, clip=deptos, cmap="Reds", extent=deptos.total_bounds)

### Leyendo el patrón espacial

Un mapa no es solo un dibujo lindo: es una herramienta para hacer preguntas. Mirando el KDE:

* ¿En qué regiones del país se concentran los focos? ¿Coinciden con algo que conozcas (zonas agrícolas, forestales, ganaderas, humedales)?
* ¿Hay departamentos con pocos focos? ¿Por qué podría ser?
* En el mapa de puntos se veían algunos **puntos que se repiten en el mismo lugar**, año tras año. ¿Puede un incendio de pastizal repetirse siempre en el mismo punto? ¿Qué otra cosa caliente y fija podría detectar un satélite? (Pista: industrias, plantas de celulosa, caleras.)

Contar también ayuda a leer el mapa:

In [ ]:
# cantidad de focos por departamento, de mayor a menor
focos["depto"].value_counts()

---
# Densidad de kernel para ver el espacio **y** el tiempo

¿La ocurrencia de focos de calor varió en el espacio a lo largo del año? Primero, miremos solo el tiempo: ¿en qué meses hay más focos?

In [ ]:
focos.columns

In [ ]:
# cantidad de focos por mes (todos los años juntos)
ax = focos.groupby("mes").size().plot.bar(color="tab:red", figsize=(8, 3))
ax.set_ylabel("cantidad de focos")
ax.set_title("Focos de calor por mes, 2018–2024")

Ahora, el espacio y el tiempo juntos. Vamos a separar los focos del **verano** (diciembre a febrero) y del **otoño** (marzo a mayo), y a comparar dónde se concentran.

In [ ]:
# misma lógica que focos_lav: una condición entre corchetes
focos_verano = focos[focos["estacion"] == "verano"]
focos_otono = focos[focos["estacion"] == "otoño"]
print(len(focos_verano), "focos en verano y", len(focos_otono), "en otoño")

Dibujamos el KDE del verano con relleno (rojo) y, encima, el KDE del otoño en isolíneas (azul). Así vemos en un solo mapa si las concentraciones se mueven.

In [ ]:
ax = deptos.plot(facecolor="none", edgecolor="grey", linewidth=0.5, figsize=(8, 8))
gplt.kdeplot(focos_verano, ax=ax, extent=deptos.total_bounds, fill=True, clip=deptos, cmap="Reds")
gplt.kdeplot(focos_otono, ax=ax, extent=deptos.total_bounds, clip=deptos, cmap="Blues")
ax.set_title("KDE de focos: verano (relleno rojo) y otoño (líneas azules)")

Otra forma es poner los mapas **lado a lado**. Con `plt.subplots(2, 2)` creamos una figura con 4 "lienzos" en una grilla de 2 × 2 (uno por estación) y, con un `for`, repetimos el mismo mapa para cada estación.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 12))
for ax, estacion in zip(axs.flat, ["verano", "otoño", "invierno", "primavera"]):
    sub = focos[focos["estacion"] == estacion]
    deptos.plot(ax=ax, facecolor="none", edgecolor="grey", linewidth=0.5)
    gplt.kdeplot(sub, ax=ax, extent=deptos.total_bounds, fill=True, clip=deptos, cmap="Reds")
    ax.set_title(f"{estacion} (n = {len(sub)})")

**Para pensar:**

* ¿El patrón es el mismo en todas las estaciones? ¿Dónde se concentran los focos en verano y dónde en invierno?
* ¿Qué actividades humanas o qué condiciones del clima podrían explicar esas diferencias? (Pensá en las quemas de campo, las cosechas, la sequía.)
* ¿Hay alguna zona que aparezca en todas las estaciones? ¿Te acordás de los puntos fijos?
* ¿Será que dichos patrones se repiten en años distintos?

> Desafío: en vez de estaciones, compará años. El año 2020 tuvo casi el doble de focos que los demás: ¿dónde se concentraron? Pista: `focos[focos["anio"] == 2020]`.

---
# **Pausa**
---
# Datos ráster o matriciales

Los datos ráster o matriciales **se usan, principalmente, para representar fenómenos que varían de forma continua en el espacio**, es decir, que no tienen una localización, un límite o una dimensión bien definidos.

Como su nombre lo indica, es un dato representado por una matriz. Los datos ráster, como las matrices matemáticas, **se usan para representar valores numéricos y no tienen una tabla de atributos con otros datos asociados**. Cada elemento de la matriz se presenta como un *píxel* (*picture element*) de una "imagen".

Si viste una imagen de satélite, ya viste un ráster. Pero los rásters con los que vamos a trabajar son un poco diferentes de una fotografía común, porque están **georreferenciados**: cada píxel representa un área geográfica. **Esa área está definida por la resolución espacial del ráster.**

* Cada *píxel* representa un área geográfica;
* La resolución espacial indica el área que representa cada píxel en el terreno.
> Si un dato tiene una resolución espacial de 10 metros, cada píxel representa un cuadrado de 10 × 10 metros en el terreno.

Los datos ráster se usan para representar muchos tipos de datos, como:
* Elevación (altimetría);
* Altura de los árboles;
* Temperatura de la superficie;
* Modelos climáticos;
* Cobertura y uso del suelo.

<img width=80% src="https://earthdatascience.org/images/courses/earth-analytics/raster-data/raster-resolution.png">

### Celda de rescate

In [ ]:
URL = "https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/datos/"
focos = gpd.read_file(URL + "focos_firms_uy.zip")
minas = gpd.read_file(URL + "taller1_uy.gpkg", layer="minas")
focos_lav = focos[focos["depto"] == "LAVALLEJA"]

# Modelo Digital del Terreno (MDT)

Es un ráster de **una sola banda** (capa).

Los rásters de una sola banda representan una única información. Un ejemplo clásico son los datos de elevación o altitud, también llamados Modelo Digital de Elevación (MDE) o, cuando representan el suelo "desnudo", sin árboles ni edificios, Modelo Digital del Terreno (MDT).

Vamos a usar el **MDT nacional de la [IDE Uruguay](https://www.gub.uy/infraestructura-datos-espaciales/)**, que tiene una resolución original de 2,5 m. Para el taller lo recortamos a Minas y alrededores y lo llevamos a 5 m, para que pese menos.

## Cargando el dato de altitud

Para leer `mdt_minas.tif` vamos a usar la función `open()` de `rasterio`. Algunos detalles:
* La función recibe la dirección del archivo;
* `open()` solo crea una **conexión** con el archivo y lee algunas informaciones importantes, pero todavía no carga los valores.

In [ ]:
mdt = rasterio.open(URL + "mdt_minas.tif")
mdt

Con la conexión abierta, podemos ver algunos parámetros del ráster: tamaño (`width` y `height`, en píxeles), cantidad de bandas (`count`), tipo de dato (`dtype`), sistema de referencia (`crs`), el valor que significa "sin dato" (`nodata`) y la `transform`, que dice dónde está ubicado y el tamaño de cada píxel.

In [ ]:
mdt.profile

Ahora sí, leemos los valores con `.read(1)` (la banda 1). Con `masked=True` le decimos que los píxeles "sin dato" no se usen en los cálculos ni en los gráficos.

In [ ]:
altitud = mdt.read(1, masked=True)
print("forma de la matriz:", altitud.shape)
print("altitud mínima:", altitud.min(), "m | máxima:", altitud.max(), "m")
altitud

Para visualizarlo, usamos la función `show()` de `rasterio.plot`. Pasándole la conexión (`mdt`), ya sabe dónde ubicar el ráster en el mapa. Con `cmap` elegimos la paleta de colores.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
show(mdt, ax=ax, cmap="terrain")
ax.set_title("Altitud en Minas y alrededores (m)")

## Sistemas de Referencia de Coordenadas (CRS)

Queremos dibujar el límite de Minas encima del MDT. Pero hay un detalle: **nuestras capas no están en el mismo sistema de coordenadas**.

In [ ]:
print("Minas:", minas.crs)
print("MDT:  ", mdt.crs)

Al final, ¿qué son la latitud y la longitud?

<img width=40% src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/83/Latitude_and_longitude_graticule_on_a_sphere.svg/500px-Latitude_and_longitude_graticule_on_a_sphere.svg.png">

* **EPSG:4326** (WGS 84) es un sistema **geográfico**: las coordenadas son ángulos (grados de longitud y latitud) sobre la superficie de la Tierra. Es el que usan el GPS y la mayoría de los datos globales, como los focos de FIRMS.
* **EPSG:5382** (SIRGAS-ROU98 / UTM zona 21 S) es el sistema **proyectado** oficial de Uruguay: la Tierra se "aplana" en un plano y las coordenadas están en **metros**. Así se pueden medir distancias y áreas.

Para dibujar capas juntas, tienen que estar en el mismo CRS. En `geopandas` se convierte con `.to_crs()`:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
show(mdt, ax=ax, cmap="terrain")
minas.to_crs(mdt.crs).boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Altitud (m) y límite de la ciudad de Minas")

**Supongamos que una crecida del arroyo San Francisco, que atraviesa Minas, puede inundar todas las zonas por debajo de 130 m de altitud.** ¿Cómo podemos ver esas zonas?

Con `np.ma.masked_where()` "escondemos" los píxeles que cumplen una condición. Si escondemos todo lo que está **por encima** de 130 m, lo que queda visible son las zonas bajas.

In [ ]:
zonas_bajas = np.ma.masked_where(altitud > 130, altitud)

fig, ax = plt.subplots(figsize=(8, 8))
show(zonas_bajas, transform=mdt.transform, ax=ax, cmap="terrain")
minas.to_crs(mdt.crs).boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_title("Zonas por debajo de 130 m")

In [ ]:
# Ejercicio: cambiá el umbral (por ejemplo, 120 o 140 m). ¿Qué barrios de Minas quedarían afectados?


# Ráster con varias bandas: imágenes de satélite

Vamos a usar una imagen del satélite **Sentinel-2**, del programa europeo [Copernicus](https://www.copernicus.eu/es), tomada el **28 de febrero de 2026** sobre Minas y alrededores (16 × 16 km). Sentinel-2 pasa por el mismo lugar cada 5 días y sus imágenes son libres y gratuitas.

Abrimos `sentinel2_minas.tif` de la misma forma que el MDT. Es un `.tif` con **varias bandas**.

In [ ]:
s2 = rasterio.open(URL + "sentinel2_minas.tif")
s2.profile

In [ ]:
imagen = s2.read()
print("forma:", imagen.shape, "→ (bandas, filas, columnas)")

## Teledetección (sensoriamento remoto)

### Espectro electromagnético

Los sensores de los satélites no "sacan fotos" como un celular: miden la energía que reflejan los objetos en distintas partes del **espectro electromagnético**, incluso en algunas que nuestros ojos no ven, como el infrarrojo.

<img width=80% src="https://seos-project.eu/remotesensing/images/Spektrum_eng_h600.png">

**Las cuatro bandas del ráster que abrimos son:**

| Índice en Python | Banda de Sentinel-2 | Región del espectro |
|---|---|---|
| 0 | B02 | azul (~490 nm) |
| 1 | B03 | verde (~560 nm) |
| 2 | B04 | rojo (~665 nm) |
| 3 | B08 | infrarrojo cercano (~842 nm) |

**Recuerden que en Python los índices empiezan en 0. Entonces, la banda 1 es la 0...**

Los valores son la **reflectancia** (qué proporción de la luz refleja la superficie) multiplicada por 10.000.

## Composición RGB

Una pantalla forma los colores combinando tres canales: rojo (R), verde (G) y azul (B). Para ver la imagen con "colores naturales", ponemos la banda roja en el canal rojo, la verde en el verde y la azul en el azul: `(2, 1, 0)`.

Para eso vamos a usar la función `plot_rgb()` del módulo [`earthpy.plot`](https://earthpy.readthedocs.io/), que importamos como `ep`. Recibe la imagen y, en el parámetro `rgb`, las tres bandas que queremos en los canales R, G y B. Primero, veamos la imagen tal como viene, sin ningún ajuste.

In [ ]:
ep.plot_rgb(imagen, rgb=(2, 1, 0), figsize=(8, 8),
            title="Sentinel-2, color natural, sin ajuste de contraste")

¿Se ve muy oscura? Es lo normal: la mayor parte de los píxeles tiene valores de reflectancia bajos, concentrados en una franja angosta, y unos pocos píxeles muy brillantes (techos, suelo desnudo) "estiran" la escala de colores. Por eso casi siempre hay que **ajustar el contraste** (estirar el histograma).

Con `stretch=True`, `plot_rgb()` descarta el 2 % de los valores más bajos y el 2 % de los más altos de cada banda (eso lo controla el parámetro `str_clip=2`) y reparte el resto en toda la escala de colores.

In [ ]:
ep.plot_rgb(imagen, rgb=(2, 1, 0), stretch=True, figsize=(8, 8),
            title="Sentinel-2, color natural, con ajuste de contraste (stretch=True)")

Hora de probar otras composiciones, ya que tenemos la banda del infrarrojo cercano (índice 3).
* ¿En qué canal la pondrían?

**Sugerencia: infrarrojo cercano (3) en el canal rojo: `(3, 2, 1)`. Es el clásico "falso color".**

In [ ]:
# Ejercicio: probá la composición (3, 2, 1), con stretch=True, y otras que se te ocurran.
# ¿Qué cambia si usás otro valor de str_clip (por ejemplo, str_clip=0.5 o str_clip=5)?


## Sobre la reflectancia de los objetos

¿Qué observaron en las distintas composiciones? ¿Por qué la vegetación se ve roja en el falso color?

La vegetación sana **absorbe** mucho rojo (lo usa para la fotosíntesis) y **refleja** muchísimo infrarrojo cercano. El agua absorbe casi todo el infrarrojo. Las zonas urbanas y el suelo desnudo reflejan parecido en todas las bandas. A esto se le llama **firma espectral**.

<img width=60% src="https://ars.els-cdn.com/content/image/3-s2.0-B9780120644773500138-f11-04-9780120644773.gif">

# Índices espectrales: el NDVI

NDVI = *Normalized Difference Vegetation Index* = Índice de Vegetación de Diferencia Normalizada.

Aprovecha justamente esa diferencia entre el rojo y el infrarrojo cercano:

$$NDVI = \frac{NIR - Rojo}{NIR + Rojo}$$

Va de −1 a 1: valores altos (cerca de 1) indican vegetación densa y vigorosa; valores cercanos a 0, suelo desnudo o zonas urbanas; valores negativos, agua.

Como el ráster es una matriz, podemos hacer la cuenta con todas las bandas de una vez: es el **álgebra de mapas**.

In [ ]:
nir = imagen[3].astype(float)   # infrarrojo cercano
rojo = imagen[2].astype(float)  # rojo
ndvi = (nir - rojo) / (nir + rojo)
ndvi

Para verlo como mapa usamos `show()` de nuevo, indicando la `transform` del ráster, la paleta (`cmap`) y los valores mínimo y máximo de la escala de colores (`vmin` y `vmax`).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
im = show(ndvi, transform=s2.transform, ax=ax, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
fig.colorbar(im.get_images()[0], ax=ax, shrink=0.7, label="NDVI")
ax.set_title("NDVI, 28/02/2026")

### Volviendo a los focos de calor

Ahora juntamos todo: ¿dónde cayeron los focos de calor de los últimos años cerca de Minas? ¿Sobre qué tipo de cobertura? Para dibujar los focos sobre la imagen, primero los llevamos al mismo CRS que el ráster.

In [ ]:
focos_s2 = focos_lav.to_crs(s2.crs)

fig, ax = plt.subplots(figsize=(9, 8))
show(ndvi, transform=s2.transform, ax=ax, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
minas.to_crs(s2.crs).boundary.plot(ax=ax, color="black", linewidth=1)
focos_s2.plot(ax=ax, column="estacion", legend=True, markersize=25, edgecolor="black")
ax.set_xlim(s2.bounds.left, s2.bounds.right)
ax.set_ylim(s2.bounds.bottom, s2.bounds.top)
ax.set_title("NDVI y focos de calor 2018–2024 cerca de Minas")

* ¿Los focos caen sobre vegetación (NDVI alto) o sobre zonas de NDVI bajo?
* Al oeste de Minas hay un grupo de focos muy juntos. Mirá esa zona en la composición de color natural y en el MDT (¿te acordás del "pozo" azul oscuro, mucho más bajo que todo lo que lo rodea?): ¿qué parece ser? ¿Es un incendio de vegetación?

## Guardando el NDVI como GeoTIFF

El `ndvi` es una matriz de `numpy`: para guardarlo **georreferenciado** tenemos que decirle a `rasterio` dónde va (la `transform` y el `crs`). La forma más simple es copiar el perfil de la imagen original y cambiar lo que es distinto: ahora es una sola banda, con números decimales.

Guardar resultados intermedios evita tener que volver a correr todo el proceso, y permite abrirlos en otros programas, como QGIS.

In [ ]:
perfil = s2.profile.copy()
perfil.update(count=1, dtype="float32", nodata=None)

with rasterio.open("ndvi_minas.tif", "w", **perfil) as destino:
    destino.write(ndvi.astype("float32"), 1)

rasterio.open("ndvi_minas.tif").profile

---
# Tu mapa (evaluación)

Llegó el momento de hacer **tu propio mapa**, con lo que aprendiste hoy. Es la entrega para la evaluación del taller.

**Consigna:**
1. Elegí un tema: por ejemplo, los focos de calor de un departamento, de una estación o de un año; un KDE; la altitud de Minas; o el NDVI con los focos.
2. Tu mapa tiene que tener **al menos dos capas** (por ejemplo, un límite + puntos, o un ráster + un límite).
3. Ponele un **título** que incluya **tu nombre**, y elegí colores que se entiendan (con leyenda o barra de colores, si corresponde).
4. Guardalo con `plt.savefig("apellido_nombre.png", dpi=150)` en la misma celda del mapa.
5. Descargalo: en el panel de **Archivos** de Colab (el ícono de carpeta, a la izquierda), hacé clic derecho sobre `apellido_nombre.png` y elegí **Descargar**.
6. Subilo al formulario de entrega, junto con **2 o 3 líneas** que cuenten qué muestra el mapa, **hasta las 23:59 del viernes 16 de octubre**:

**[https://forms.gle/AF5s89nZzDozbyVj6](https://forms.gle/AF5s89nZzDozbyVj6)**

<img width=200 src="https://raw.githubusercontent.com/FelipeSBarros/taller-sig-python-utec-2026/main/img/qr_formulario.png">

**Criterios:** que tenga al menos dos capas, título con tu nombre, colores y leyenda adecuados, y una interpretación breve y coherente.

In [ ]:
# Tu mapa: escribí aquí el código. Podés partir de cualquier celda anterior.
# fig, ax = plt.subplots(figsize=(8, 8))
# ...
# ax.set_title("... | Nombre Apellido")
# plt.savefig("apellido_nombre.png", dpi=150)


---
# ¿Dónde encontrar datos geográficos de Uruguay?

* [Observatorio Ambiental Nacional (OAN)](https://www.ambiente.gub.uy/oan/): datos ambientales del Ministerio de Ambiente (calidad de agua, áreas protegidas, emprendimientos con autorización ambiental...).
* [IDE Uruguay](https://visualizador.ide.uy/): cartografía oficial, ortoimágenes y modelos de elevación de todo el país.
* [Catálogo de Datos Abiertos](https://catalogodatos.gub.uy/): datos de todo el Estado; muchos tienen coordenadas.
* [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/): focos de calor de todo el mundo, casi en tiempo real.
* [Copernicus Browser](https://browser.dataspace.copernicus.eu/): imágenes Sentinel, para ver y descargar.
* [Earth Search](https://element84.com/earth-search/): catálogo (STAC) de imágenes Sentinel-2 y Landsat listas para el análisis, de donde sacamos la imagen de hoy.

# Algunas personas interesantes para seguir

* [Guillermo D'Angelo](https://www.linkedin.com/in/guillermodangelo/)
* [Geoff Boeing](https://geoffboeing.com/);
* [Jakub Nowosad](https://jakubnowosad.com/);
* [Anita Graser](https://anitagraser.com/).